# **Objetivo del Notebook**

Este primer notebook tiene como propósito realizar una exploración descriptiva inicial de la base de datos, caracterizando el comportamiento de las variables numéricas y categóricas mediante estadísticas descriptivas, tablas de frecuencia y visualizaciones gráficas. Este análisis permite obtener una comprensión preliminar de la estructura de los datos, identificar patrones relevantes y evaluar la calidad de la información disponible.

Adicionalmente, el notebook funciona como una etapa de diagnóstico y control de calidad de los datos, orientada a detectar posibles problemas que puedan afectar análisis posteriores, tales como valores atípicos, datos faltantes, inconsistencias en la codificación, categorías con baja representatividad, desbalance de clases y otras anomalías en la distribución de las variables.

Con base en la evidencia obtenida durante esta exploración, se plantean y justifican las primeras decisiones de limpieza, transformación y depuración de la información, garantizando que las etapas posteriores de análisis estadístico y modelamiento se desarrollen sobre una base de datos más consistente, confiable y adecuada para los objetivos del proyecto.


## **Librerías**

In [42]:
import numpy as np
import pandas as pd
import chardet
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import json
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
import unicodedata
import folium
import os

## **Cargar el dataset**

In [43]:
# ============================================================
# CARGA DE DATOS
# ============================================================

ruta = "../data/raw/Nacimientos_BPN.csv"

datos = pd.read_csv(
    ruta,
    sep=";",
    encoding="ISO-8859-1"
)

# ============================================================
# VERIFICACIÓN INICIAL
# ============================================================

print(f"Dimensiones: {datos.shape}")
datos.head()

C:\Users\Usuario\AppData\Local\Temp\ipykernel_6132\1051411502.py:7: DtypeWarning:

Columns (10,15,16,22) have mixed types. Specify dtype option on import or set low_memory=False.



Dimensiones: (384321, 39)


,COD_LOCALIDAD_NACIMIENTO,LOCALIDAD_NACIMIENTO,SIT_PARTO,SEXO,PESO_GRAMOS,PESO,BPN,TALLA_CENTIMETROS,ANO,ATENCION_PARTO_POR,...,NUM_HIJOS_NACIDOS_VIVOS,FECHA_ANTERIOR_HIJO_NACIDO_VIVO,NUM_EMBARAZOS,REGIMEN_SEGURIDAD,EDAD_PADRE,GRUPO_QUINQUENAL_PADRE,GRUPO_QUINQUENAL_PADRE_CALCULADORA,NIVEL_EDUCATIVO_PADRE,ULTIMO_CURSO_APROBADO_PADRE,BPN_1
0,13,Teusaquillo,1.INSTITUCION DE SALUD,MASCULINO,3330.0,5.NORMAL(3000_4199),0,51,2020,1.MEDICO,...,2.0,28/09/2009,2.0,CONTRIBUTIVO,34.0,NaN,08. 30 a 34 años,04.MEDIA ACADEMICA,11.0,0.0
1,8,Kennedy,1.INSTITUCION DE SALUD,FEMENINO,2625.0,4.DEFICIT(2500_2999),0,47,2020,1.MEDICO,...,1.0,NaN,1.0,NO ASEGURADO,21.0,NaN,06. 20 a 24 años,04.MEDIA ACADEMICA,11.0,0.0
2,9,Fontibón,1.INSTITUCION DE SALUD,FEMENINO,3150.0,5.NORMAL(3000_4199),0,49,2020,1.MEDICO,...,3.0,27/04/2018,3.0,SUBSIDIADO,36.0,NaN,09. 35 a 39 años,04.MEDIA ACADEMICA,11.0,0.0
3,2,Chapinero,1.INSTITUCION DE SALUD,MASCULINO,4475.0,6.EXCESO(MAS4200),0,54,2020,1.MEDICO,...,1.0,NaN,1.0,CONTRIBUTIVO,44.0,NaN,10. 40 a 44 años,09.PROFESIONAL,5.0,0.0
4,12,Barrios Unidos,1.INSTITUCION DE SALUD,FEMENINO,2380.0,3.BAJO(1500_2499),1,48,2020,1.MEDICO,...,1.0,NaN,1.0,CONTRIBUTIVO,25.0,NaN,07. 25 a 29 años,04.MEDIA ACADEMICA,11.0,1.0


## **Campos iniciales**

In [ ]:
# =========================================================
# TIPOS DE VARIABLES
# =========================================================
tipos_variables = (
    datos.dtypes
    .astype(str)
    .reset_index()
)

tipos_variables.columns = ['Variable', 'Tipo']

# Mostrar tabla
display(tipos_variables)

# =========================================================
# RESUMEN DE CANTIDAD POR TIPO
# =========================================================
resumen_tipos = (
    tipos_variables['Tipo']
    .value_counts()
    .reset_index()
)

resumen_tipos.columns = ['Tipo', 'Cantidad']

# Proporción
resumen_tipos['Proporcion (%)'] = (
    resumen_tipos['Cantidad'] / resumen_tipos['Cantidad'].sum()
) * 100

display(resumen_tipos)

# =========================================================
# GRÁFICO DE TORTA
# =========================================================
#plt.figure(figsize=(8, 8))

#plt.pie(
#    resumen_tipos['Cantidad'],
#    labels=resumen_tipos['Tipo'],
#    autopct='%1.1f%%',
#    startangle=90
#)

#plt.title('Proporción de Tipos de Variables')
#plt.tight_layout()
#plt.show()

# **Identificación de datos faltante**

## **Datos faltantes en general**

In [ ]:
# =========================================================
# TABLA RESUMEN DE NULOS
# =========================================================

total_nulos = datos.isnull().sum()

porcentaje_nulos = (
    total_nulos / len(datos)
) * 100

datos_faltantes = pd.concat(
    [total_nulos, porcentaje_nulos],
    axis=1,
    keys=[
        'Total de nulos',
        'Porcentaje de nulos (%)'
    ]
)

# Mostrar tabla completa
#display(datos_faltantes)

# =========================================================
# FILTRAR VARIABLES CON NULOS
# =========================================================

datos_faltantes_graf = (
    datos_faltantes[
        datos_faltantes['Total de nulos'] > 0
    ]
    .sort_values(
        'Porcentaje de nulos (%)',
        ascending=True
    )
    .reset_index()
    .rename(columns={'index': 'Variable'})
)

# =========================================================
# GRÁFICO DE BARRAS HORIZONTALES
# =========================================================

plt.figure(
    figsize=(12, max(6, len(datos_faltantes_graf) * 0.45))
)

# Paleta gradual
colores = sns.color_palette(
    "YlOrRd",
    n_colors=len(datos_faltantes_graf)
)

bars = plt.barh(
    datos_faltantes_graf['Variable'],
    datos_faltantes_graf['Porcentaje de nulos (%)'],
    color=colores,
    edgecolor='black'
)

# =========================================================
# ETIQUETAS DE PORCENTAJE
# =========================================================

for bar in bars:

    ancho = bar.get_width()

    plt.text(
        ancho + 0.3,
        bar.get_y() + bar.get_height()/2,
        f'{ancho:.2f}%',
        va='center',
        fontsize=9
    )

# =========================================================
# LÍNEAS DE REFERENCIA
# =========================================================

plt.axvline(
    x=5,
    color='orange',
    linestyle='--',
    linewidth=2,
    label='5%'
)

plt.axvline(
    x=15,
    color='red',
    linestyle='--',
    linewidth=2,
    label='15%'
)

# =========================================================
# FORMATO
# =========================================================

plt.title(
    'Porcentaje de Valores Nulos por Variable',
    fontsize=15,
    fontweight='bold'
)

plt.xlabel('Porcentaje de nulos (%)')
plt.ylabel('')

plt.grid(
    axis='x',
    linestyle=':',
    alpha=0.5
)

plt.legend()

plt.tight_layout()

plt.show()

## **Datos faltantes desagregados por año**

In [ ]:
# =========================================================
# CALCULAR NULOS POR VARIABLE Y AÑO
# =========================================================

col_anio = 'ANO'

resultados = []

for col in datos.columns:

    if col == col_anio:
        continue

    resumen = (
        datos
        .groupby(col_anio)
        .agg(
            total=(col, 'size'),
            nulos=(col, lambda x: x.isnull().sum())
        )
        .reset_index()
    )

    resumen['proporcion_nulos'] = resumen['nulos'] / resumen['total']
    resumen['porcentaje_nulos'] = resumen['proporcion_nulos'] * 100
    resumen['variable'] = col

    resultados.append(resumen)

df_nulos = pd.concat(resultados, ignore_index=True)

# =========================================================
# VARIABLES QUE PRESENTAN NULOS
# =========================================================

variables_con_nulos = (
    df_nulos
    .groupby('variable')['nulos']
    .sum()
    .reset_index()
)

variables_con_nulos = variables_con_nulos[
    variables_con_nulos['nulos'] > 0
]['variable'].tolist()

df_nulos_filtrado = (
    df_nulos[
        df_nulos['variable'].isin(variables_con_nulos)
    ]
    .copy()
)

# =========================================================
# AÑO CON MAYOR PROPORCIÓN DE NULOS
# =========================================================

df_max = (
    df_nulos_filtrado
    .sort_values(
        ['variable', 'proporcion_nulos'],
        ascending=[True, False]
    )
    .groupby('variable')
    .first()
    .reset_index()
)

print("\nNULOS POR VARIABLE Y AÑO")
display(df_nulos_filtrado)

print("\nAÑO CON MAYOR PROPORCIÓN DE NULOS POR VARIABLE")
display(
    df_max[
        ['variable', col_anio,
         'porcentaje_nulos',
         'nulos',
         'total']
    ]
)




# **Limpieza y transformación preliminar**

## **Conversión a tipo de variable adecuado.**

In [ ]:
# =========================================================
# COPIA DEL DATASET
# =========================================================

df = datos.copy()

# =========================================================
# CONVERSIÓN A NUMÉRICO
# =========================================================

cols_a_numerico = [
    'TIEMPO_GESTACION'
]

for col in cols_a_numerico:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# =========================================================
# VARIABLES CATEGÓRICAS
# =========================================================

cols_a_categoria = [
    'LOCALIDAD_NACIMIENTO',
    'SIT_PARTO',
    'SEXO',
    'ATENCION_PARTO_POR',
    'TIPO_PARTO',
    'MULTIPLICIDAD_PARTO',
    'GRUPO_SANGUINEO',
    'FACTOR_RH',
    'PERTENENCIA_ETNICA',
    'TIPO_DOC_MADRE',
    'GRUPO_QUINQUENAL_MADRE',
    'GRUPO_QUINQUENAL_MADRE_CALCULADORA',
    'ESTADO_CONYUGAL_MADRE',
    'NIVEL_EDUCATIVO_MADRE',
    'LOCALIDAD_MADRE',
    'REGIMEN_SEGURIDAD',
    'GRUPO_QUINQUENAL_PADRE_CALCULADORA',
    'NIVEL_EDUCATIVO_PADRE',
    'BPN',
    'APGAR1',
    'APGAR2',
    'GESTACION',
    'ANO',
    'COD_LOCALIDAD_NACIMIENTO',
    'COD_LOCALIDAD_MADRE',
    'PESO'
]

for col in cols_a_categoria:
    if col in df.columns:
        df[col] = df[col].astype('category')

# =========================================================
# TABLA DE TIPOS DE VARIABLES
# =========================================================

tipos_variables = (
    df.dtypes
    .astype(str)
    .reset_index()
)

tipos_variables.columns = ['Variable', 'Tipo']

print("\nTIPOS DE VARIABLES")
display(tipos_variables)

# =========================================================
# RESUMEN POR TIPO
# =========================================================

resumen_tipos = (
    tipos_variables['Tipo']
    .value_counts()
    .reset_index()
)

resumen_tipos.columns = ['Tipo', 'Cantidad']

resumen_tipos['Proporcion (%)'] = (
    resumen_tipos['Cantidad']
    / resumen_tipos['Cantidad'].sum()
    * 100
)

print("\nRESUMEN DE TIPOS DE VARIABLES")
display(resumen_tipos)

## **Eliminación de variables por datos faltantes y redundantes**


In [ ]:

# ---------------------------------------------------------
# VARIABLES A ELIMINAR POR CRITERIO TEÓRICO
# ---------------------------------------------------------
variables_data_leakage = [
    'BPN_1',

]

# ---------------------------------------------------------
# IDENTIFICAR VARIABLES CON >5% DE NULOS
# ---------------------------------------------------------
porcentaje_nulos = (df.isnull().mean()) * 100

variables_muchos_nulos = (
    porcentaje_nulos[porcentaje_nulos > 5]
    .index
    .tolist()
)

# ---------------------------------------------------------
# CONSOLIDAR VARIABLES A ELIMINAR
# ---------------------------------------------------------
variables_eliminar = list(
    set(variables_data_leakage + variables_muchos_nulos)
)

print("\nVARIABLES ELIMINADAS:")
print(variables_eliminar)

# ---------------------------------------------------------
# ELIMINAR VARIABLES DEL DATAFRAME
# ---------------------------------------------------------
df = df.drop(
    columns=variables_eliminar,
    errors='ignore'
)

# ---------------------------------------------------------
# OUTPUT FINAL
# ---------------------------------------------------------
print(f"\nDimensión final del dataset: {df.shape}")

# **Análisis exploratorios/EDA**

# **Boxplots y distribuciones**
Para comenzar el análisis exploratorio, se graficaron las distribuciones y boxplots de las numéricas en cuestión.

In [ ]:
# Títulos legibles por variable
titulos = {
    'PESO_GRAMOS': 'Peso al nacer (gramos)',
   'TALLA_CENTIMETROS': 'Talla al nacer (cm)',
    'NUM_CONSULTAS_PRENAT': 'Número de controles prenatales',
    'EDAD_MADRE': 'Edad de la madre',
    'ULTIMO_CURSO_APROBADO_MADRE': 'Último curso aprobado (madre)',
    'NUM_HIJOS_NACIDOS_VIVOS': 'Número de hijos nacidos vivos',
    'NUM_EMBARAZOS': 'Número de embarazos',
    'EDAD_PADRE': 'Edad del padre',
    'ULTIMO_CURSO_APROBADO_PADRE': 'Último curso aprobado (padre)',
    'TIEMPO_GESTACION': 'Tiempo de gestación (semanas)',
}

variables_cuantitativas= ['PESO_GRAMOS',  'NUM_CONSULTAS_PRENAT',
                          'EDAD_MADRE', 'ULTIMO_CURSO_APROBADO_MADRE',  'NUM_HIJOS_NACIDOS_VIVOS',
                          'NUM_EMBARAZOS', 'EDAD_PADRE', 'ULTIMO_CURSO_APROBADO_PADRE', 'TIEMPO_GESTACION', 'TALLA_CENTIMETROS'] 
# Lista para almacenar todas las estadísticas
resumen_estadisticas = []

# Recorrer cada variable
for var in variables_cuantitativas:

    if var not in df.columns:
        print(f"\nLa variable '{var}' no existe en df.\n")
        continue

    datos_validos = df[var].dropna()
    total = len(datos_validos)

    if total == 0:
        print(f"\nLa variable '{var}' no tiene datos válidos.\n")
        continue

    # Estadísticas
    media = datos_validos.mean()
    mediana = datos_validos.median()
    std = datos_validos.std()
    minimo = datos_validos.min()
    maximo = datos_validos.max()
    p25 = datos_validos.quantile(0.25)
    p75 = datos_validos.quantile(0.75)
    nulos = df[var].isnull().sum()

    # Guardar en resumen
    resumen_estadisticas.append({
        'Variable': var,
        'Nombre': titulos.get(var, var),
        'Total válidos': total,
        'Media': round(media, 2),
        'Mediana': round(mediana, 2),
        'Desviación estándar': round(std, 2),
        'Mínimo': minimo,
        'P25': p25,
        'P75': p75,
        'Máximo': maximo,
        'Valores nulos': nulos
    })

    # Imprimir estadísticas
#    print(f"\nEstadísticas descriptivas para '{titulos.get(var, var)}' ({var}):\n")
#    print(f"Total registros válidos: {total}")
#    print(f"Media: {media:.2f}")
#    print(f"Mediana: {mediana:.2f}")
#    print(f"Desviación estándar: {std:.2f}")
#    print(f"Mínimo: {minimo}")
#    print(f"P25: {p25}")
#    print(f"P75: {p75}")
#    print(f"Máximo: {maximo}")
#    print(f"Valores nulos: {nulos}")

    # Crear subplots: histograma + boxplot
    fig = make_subplots(
        rows=1,
        cols=2,
        horizontal_spacing=0.10,
        subplot_titles=(
            f"Distribución de {titulos.get(var, var)}",
            f"Boxplot de {titulos.get(var, var)}"
        )
    )

    # Histograma
    fig.add_trace(
        go.Histogram(
            x=datos_validos,
            nbinsx=30,
            name='Frecuencia'
        ),
        row=1,
        col=1
    )

    # Boxplot
    fig.add_trace(
        go.Box(
            x=datos_validos,
            name=titulos.get(var, var),
            boxpoints='outliers'
        ),
        row=1,
        col=2
    )

    # Layout responsivo
    fig.update_layout(
        title_text=f"Análisis descriptivo de {titulos.get(var, var)}",
        autosize=True,
        height=500,
        showlegend=False,
        template='plotly_white',
        margin=dict(l=40, r=40, t=80, b=50)
    )

    fig.update_xaxes(title_text=titulos.get(var, var), row=1, col=1)
    fig.update_yaxes(title_text='Frecuencia', row=1, col=1)
    fig.update_xaxes(title_text=titulos.get(var, var), row=1, col=2)

    fig.show(config={'responsive': True})

# DataFrame resumen final
df_resumen_estadisticas = pd.DataFrame(resumen_estadisticas)

print("\nResumen consolidado de estadísticas descriptivas:\n")
display(df_resumen_estadisticas)

Las variables cuantitativas presentan, en general, distribuciones coherentes con la población de nacimientos analizada. El peso al nacer muestra una media de 2.936 g y una mediana de 2.980 g, lo que indica una ligera asimetría hacia valores bajos asociada a la presencia de recién nacidos con bajo peso; asimismo, la desviación estándar de 508 g refleja una variabilidad moderada dentro de la cohorte. La edad materna se concentra principalmente entre los 23 y 32 años (P25=23; P75=32), mientras que la edad paterna presenta valores ligeramente superiores (mediana de 30 años). El número de controles prenatales evidencia una adecuada cobertura asistencial, con una mediana de 7 controles y el 75% de las gestantes registrando hasta 8 controles. En cuanto a las variables obstétricas, la mediana de edad gestacional fue de 38 semanas, consistente con el predominio de nacimientos a término observado previamente. Debe destacarse que varias variables contienen valores máximos de 99 o 9999, los cuales corresponden a códigos administrativos de “sin información” y no a observaciones reales; por tanto, estos valores deberán ser tratados como faltantes antes de cualquier análisis inferencial o modelamiento para evitar sesgos en las estimaciones y distorsiones en las distribuciones observadas. 


## **Análisis univariados desagregados por año para variables cuantitativas**

In [ ]:
# ============================================================
# DESCRIPTIVOS POR AÑO
# ============================================================
# Paletas para alternar colores
paleta_base = px.colors.qualitative.Set2
paleta_secundaria = px.colors.qualitative.Pastel
paleta_box = px.colors.qualitative.Safe
paleta_target = px.colors.qualitative.Dark2

resumen_anual = []

for var in variables_cuantitativas:

    df_temp = df[[col_anio, var]].dropna()

    if df_temp.empty:
        continue

    resumen = (
        df_temp
        .groupby(col_anio)[var]
        .agg(
            n='count',
            media='mean',
            mediana='median',
            std='std',
            minimo='min',
            p25=lambda x: x.quantile(0.25),
            p75=lambda x: x.quantile(0.75),
            maximo='max'
        )
        .reset_index()
    )

    resumen['variable'] = var
    resumen['nombre_variable'] = titulos.get(var, var)

    resumen_anual.append(resumen)

df_resumen_anual = pd.concat(resumen_anual, ignore_index=True)

print("\nDESCRIPTIVOS POR AÑO:")
display(df_resumen_anual)

# ============================================================
#  GRÁFICOS GLOBALES: HISTOGRAMA + BOXPLOT
# ============================================================

for i, var in enumerate(variables_cuantitativas):

    datos_validos = df[var].dropna()

    if len(datos_validos) == 0:
        continue

    color_hist = paleta_base[i % len(paleta_base)]
    color_box = paleta_secundaria[i % len(paleta_secundaria)]

    fig = make_subplots(
        rows=1,
        cols=2,
        horizontal_spacing=0.10,
        subplot_titles=(
            f"Distribución de {titulos.get(var, var)}",
            f"Boxplot de {titulos.get(var, var)}"
        )
    )

    # Histograma
    fig.add_trace(
        go.Histogram(
            x=datos_validos,
            nbinsx=30,
            name='Frecuencia',
            marker_color=color_hist,
            opacity=0.85
        ),
        row=1,
        col=1
    )

    # Boxplot
    fig.add_trace(
        go.Box(
            x=datos_validos,
            name=titulos.get(var, var),
            boxpoints='outliers',
            marker_color=color_box
        ),
        row=1,
        col=2
    )

    fig.update_layout(
        title_text=f"Análisis descriptivo global: {titulos.get(var, var)}",
        autosize=True,
        height=500,
        showlegend=False,
        template='plotly_white',
        margin=dict(l=40, r=40, t=80, b=50)
    )

    fig.update_xaxes(title_text=titulos.get(var, var), row=1, col=1)
    fig.update_yaxes(title_text='Frecuencia', row=1, col=1)
    fig.update_xaxes(title_text=titulos.get(var, var), row=1, col=2)

    fig.show(config={'responsive': True})

# ============================================================
#  BOXPLOT POR AÑO
# ============================================================

for i, var in enumerate(variables_cuantitativas):

    if df[var].dropna().empty:
        continue

    fig = px.box(
        df,
        x=col_anio,
        y=var,
        title=f"{titulos.get(var, var)} por año",
        points='outliers',
        color=col_anio,
        color_discrete_sequence=paleta_box
    )

    fig.update_layout(
        autosize=True,
        height=550,
        showlegend=False,
        template='plotly_white',
        margin=dict(l=40, r=40, t=80, b=50),
        xaxis_title='Año',
        yaxis_title=titulos.get(var, var)
    )

    fig.show(config={'responsive': True})

# ============================================================
#  TENDENCIA DE LA MEDIA POR AÑO
# ============================================================

for i, var in enumerate(variables_cuantitativas):

    df_temp = df[[col_anio, var]].dropna()

    if df_temp.empty:
        continue

    resumen_media = (
        df_temp
        .groupby(col_anio)[var]
        .mean()
        .reset_index()
    )

    fig = px.line(
        resumen_media,
        x=col_anio,
        y=var,
        markers=True,
        title=f"Tendencia temporal de la media: {titulos.get(var, var)}"
    )

    fig.update_traces(
        line=dict(width=3),
        marker=dict(size=8)
    )

    fig.update_layout(
        autosize=True,
        height=550,
        template='plotly_white',
        margin=dict(l=40, r=40, t=80, b=50),
        xaxis_title='Año',
        yaxis_title=f"Media de {titulos.get(var, var)}"
    )

    fig.show(config={'responsive': True})

# ============================================================
#  HISTOGRAMAS FACETADOS POR AÑO
# ============================================================

for i, var in enumerate(variables_cuantitativas):

    df_temp = df[[col_anio, var]].dropna()

    if df_temp.empty:
        continue

    fig = px.histogram(
        df_temp,
        x=var,
        facet_col=col_anio,
        facet_col_wrap=4,
        nbins=30,
        title=f"Distribución de {titulos.get(var, var)} por año",
        opacity=0.85
    )

    fig.update_layout(
        autosize=True,
        height=900,
        template='plotly_white',
        showlegend=False,
        margin=dict(l=40, r=40, t=80, b=50)
    )

    fig.for_each_annotation(
        lambda a: a.update(text=a.text.split("=")[-1])
    )

    fig.update_xaxes(title_text=titulos.get(var, var))
    fig.update_yaxes(title_text='Frecuencia')

    fig.show(config={'responsive': True})

# ============================================================
#  BOXPLOTS POR AÑO SEGÚN BPN
# ============================================================

col_target = 'BPN'

if col_target in df.columns:

    for i, var in enumerate(variables_cuantitativas):

        df_temp = df[[col_anio, var, col_target]].dropna()

        if df_temp.empty:
            continue

        fig = px.box(
            df_temp,
            x=col_anio,
            y=var,
            color=col_target,
            title=f"{titulos.get(var, var)} por año según {titulos.get(col_target, col_target)}",
            color_discrete_sequence=paleta_target,
            points='outliers'
        )

        fig.update_layout(
            autosize=True,
            height=600,
            template='plotly_white',
            margin=dict(l=40, r=40, t=80, b=50),
            xaxis_title='Año',
            yaxis_title=titulos.get(var, var),
            legend_title=titulos.get(col_target, col_target)
        )

        fig.show(config={'responsive': True})

# ============================================================
#  TABLA DE PREVALENCIA DE BPN POR AÑO
# ============================================================

if col_target in df.columns:

    tabla_bpn_anual = (
        df[[col_anio, col_target]]
        .dropna()
        .groupby([col_anio, col_target])
        .size()
        .reset_index(name='conteo')
    )

    total_anual = (
        tabla_bpn_anual
        .groupby(col_anio)['conteo']
        .sum()
        .reset_index(name='total_anual')
    )

    tabla_bpn_anual = tabla_bpn_anual.merge(
        total_anual,
        on=col_anio,
        how='left'
    )

    tabla_bpn_anual['proporcion'] = (
        tabla_bpn_anual['conteo'] /
        tabla_bpn_anual['total_anual']
    )

    print("\nPREVALENCIA DE BPN POR AÑO:")
    display(tabla_bpn_anual)

    fig = px.bar(
        tabla_bpn_anual,
        x=col_anio,
        y='proporcion',
        color=col_target,
        barmode='group',
        title='Proporción de BPN por año',
        color_discrete_sequence=paleta_target
    )

    fig.update_layout(
        autosize=True,
        height=550,
        template='plotly_white',
        margin=dict(l=40, r=40, t=80, b=50),
        xaxis_title='Año',
        yaxis_title='Proporción'
    )

    fig.show(config={'responsive': True})

# ============================================================
#  SALIDAS FINALES CONSOLIDADAS
# ============================================================

print("\nTABLAS DISPONIBLES GENERADAS:")
print("- tabla_nulos_global")
print("- df_nulos_anual")
print("- df_max_nulos")
print("- df_resumen_global")
print("- df_resumen_anual")

if col_target in df.columns:
    print("- tabla_bpn_anual")

## **Caracterización Espacial y Temporal del Peso al Nacer y BPN**

Este bloque transforma una cartografía de localidades de Bogotá almacenada en ESRI JSON a un formato GeoJSON estándar, construye las geometrías mediante shapely, las carga en un GeoDataFrame de geopandas y genera una copia en CSV de sus atributos. El resultado final (geo_bogota) constituye la capa geográfica base utilizada posteriormente para unir indicadores estadísticos por localidad y construir mapas coropléticos o análisis espaciales.

In [ ]:
# ============================================================
# CONVERSIÓN DE CARTOGRAFÍA DE LOCALIDADES DE BOGOTÁ
# DESDE FORMATO ESRI JSON A GEOJSON Y GEODATAFRAME
# ============================================================
ruta_esri = "../data/geo/localidades_bogota_esri.json"

with open(ruta_esri, "r", encoding="utf-8") as f:
    esri = json.load(f)

features_geojson = []

for feat in esri["features"]:
    attrs = feat["attributes"]
    rings = feat["geometry"]["rings"]

    polygons = []
    for ring in rings:
        polygons.append(Polygon(ring))

    geom = MultiPolygon(polygons) if len(polygons) > 1 else polygons[0]

    features_geojson.append({
        "type": "Feature",
        "properties": attrs,
        "geometry": geom.__geo_interface__
    })

geojson = {
    "type": "FeatureCollection",
    "features": features_geojson
}

ruta_geojson = "../data/geo/localidades_bogota.geojson"

with open(ruta_geojson, "w", encoding="utf-8") as f:
    json.dump(geojson, f, ensure_ascii=False)

geo_bogota = gpd.read_file(ruta_geojson)

print(geo_bogota.columns)
display(geo_bogota.head())
geo_bogota.to_csv("../data/geo/localidades_bogota.csv", index=False)


Este bloque tiene como objetivo construir una capa analítica espacial que permita caracterizar la distribución geográfica del peso al nacer y del bajo peso al nacer (BPN) según la localidad de residencia materna en Bogotá.

Inicialmente se realiza un proceso de limpieza y homologación de nombres de localidades tanto en la base de nacimientos como en la cartografía oficial de Bogotá. Esta etapa busca garantizar la correcta correspondencia entre los registros individuales y las unidades geográficas utilizadas para la representación espacial.



A nivel de localidad se calculan los siguientes indicadores descriptivos:

* **Peso promedio al nacer:** media de los pesos registrados en la localidad.
* **Mediana del peso al nacer:** medida robusta de tendencia central.
* **Prevalencia de bajo peso al nacer (BPN):** proporción de nacimientos con peso inferior a 2500 gramos.
* **Total de nacimientos registrados:** número de observaciones disponibles por localidad.

Formalmente, para cada localidad (j):

$$
\overline{Peso}*j =
\frac{1}{n_j}
\sum*{i=1}^{n_j}
Peso_{ij}
$$

$$
Prev(BPN)*j =
\frac{\sum*{i=1}^{n_j} BPN_{ij}}
{n_j}
$$

donde (n_j) representa el número de nacimientos observados en la localidad (j).

Una vez calculados los indicadores, estos se integran con la cartografía oficial mediante una unión espacial basada en los nombres homologados de las localidades. Esto permite construir mapas coropléticos donde la intensidad del color representa el valor del indicador analizado.

El bloque genera tres tipos de productos cartográficos:

### 1. Mapas estáticos

Se construyen mapas coropléticos para:

* Peso promedio al nacer.
* Prevalencia de bajo peso al nacer.
* Total de nacimientos registrados.

Estos mapas permiten identificar patrones espaciales globales y posibles diferencias territoriales entre localidades.

### 2. Mapas espacio-temporales por año

Se calculan nuevamente los indicadores para cada combinación de localidad y año de ocurrencia del nacimiento, obteniendo:

$$
\overline{Peso}_{j,t}
$$

$$
Prev(BPN)_{j,t}
$$

$$
N_{j,t}
$$

donde (t) representa el año de observación.

Posteriormente se generan paneles comparativos que permiten evaluar la evolución temporal de los indicadores y detectar cambios espaciales a lo largo del período de estudio.

### 3. Mapas interactivos

Se construyen visualizaciones interactivas mediante Folium y Plotly. Estas herramientas permiten explorar cada localidad de forma dinámica, consultando indicadores específicos mediante ventanas emergentes  y facilitando la inspección detallada de los resultados.

En conjunto, este bloque proporciona una caracterización espacial y espacio-temporal de los nacimientos registrados en Bogotá, permitiendo identificar heterogeneidad territorial en el peso al nacer, la prevalencia de bajo peso al nacer y la distribución de los nacimientos entre localidades.


In [ ]:
# ============================================================
# PIPELINE ESPACIAL  - PESO AL NACER / BPN
# ============================================================
# Requiere:
#   - df: DataFrame principal de nacimientos
#   - geo_bogota: GeoDataFrame con geometría de localidades
# ============================================================


FIG_DIR = "../reports/figures"
os.makedirs(FIG_DIR, exist_ok=True)

COL_LOCALIDAD_DATOS = "LOCALIDAD_MADRE"
COL_LOCALIDAD_GEO = "LocNombre"
COL_ANIO = "ANO"
COL_PESO = "PESO_GRAMOS"
COL_BPN = "BPN"

# ============================================================
#  FUNCIÓN DE LIMPIEZA DE TEXTO
# ============================================================

def limpiar_texto(x):
    """
    Normaliza nombres:
    - convierte a mayúsculas
    - elimina tildes
    - elimina espacios extremos
    """
    if pd.isna(x):
        return np.nan

    x = str(x).upper().strip()

    x = "".join(
        c for c in unicodedata.normalize("NFD", x)
        if unicodedata.category(c) != "Mn"
    )

    return x


# ============================================================
#  LIMPIEZA Y HOMOLOGACIÓN DE LOCALIDADES 
# ============================================================

df = df.copy()
geo_bogota = geo_bogota.copy()

df["LOCALIDAD_MADRE_LIMPIA"] = (
    df[COL_LOCALIDAD_DATOS]
    .astype("object")
    .apply(limpiar_texto)
)

geo_bogota["LOCALIDAD_LIMPIA"] = (
    geo_bogota[COL_LOCALIDAD_GEO]
    .astype("object")
    .apply(limpiar_texto)
)

reemplazos_localidad = {
    "LA CANDELARIA": "CANDELARIA"
}

df["LOCALIDAD_MADRE_LIMPIA"] = (
    df["LOCALIDAD_MADRE_LIMPIA"]
    .astype("object")
    .map(lambda x: reemplazos_localidad.get(x, x))
)


# ============================================================
#  CREACIÓN DE VARIABLE BPN NUMÉRICA 
# ============================================================
# Se calcula solo para análisis descriptivo espacial.
# Si BPN ya existe como 0/1, se convierte a numérico.
# Si no, se deriva desde PESO_GRAMOS < 2500.

if COL_BPN in df.columns:
    df["BPN_NUM"] = pd.to_numeric(df[COL_BPN], errors="coerce")
else:
    df["BPN_NUM"] = np.where(df[COL_PESO] < 2500, 1, 0)


# ============================================================
#  CÁLCULO DE INDICADORES ESPACIALES
# ============================================================
# Por localidad se calcula:
#   - peso_promedio: media de PESO_GRAMOS
#   - mediana_peso: mediana de PESO_GRAMOS
#   - prevalencia_bpn: proporción de nacimientos con BPN
#   - total_nacimientos: número de nacimientos registrados

df_indicadores_localidad = (
    df
    .dropna(subset=["LOCALIDAD_MADRE_LIMPIA"])
    .groupby("LOCALIDAD_MADRE_LIMPIA")
    .agg(
        peso_promedio=(COL_PESO, "mean"),
        mediana_peso=(COL_PESO, "median"),
        prevalencia_bpn=("BPN_NUM", "mean"),
        total_nacimientos=(COL_PESO, "count")
    )
    .reset_index()
)

display(df_indicadores_localidad.sort_values("peso_promedio"))


# ============================================================
#  MERGE GEOGRÁFICO
# ============================================================

geo_indicadores = geo_bogota.merge(
    df_indicadores_localidad,
    left_on="LOCALIDAD_LIMPIA",
    right_on="LOCALIDAD_MADRE_LIMPIA",
    how="left"
)

# Punto representativo para ubicar etiquetas de localidad
geo_indicadores["label_point"] = geo_indicadores.geometry.representative_point()


# ============================================================
#  FUNCIÓN PARA MAPA ESTÁTICO
# ============================================================

def mapa_estatico_localidad(
    geo_df,
    columna,
    titulo,
    nombre_archivo,
    cmap="viridis",
    label_col="LOCALIDAD_LIMPIA"
):
    """
    Genera mapa coroplético estático con etiquetas de localidades.
    Muestra el mapa en notebook y lo guarda como PNG.
    """

    fig, ax = plt.subplots(figsize=(12, 12))

    geo_df.plot(
        column=columna,
        cmap=cmap,
        linewidth=0.8,
        edgecolor="black",
        legend=True,
        ax=ax,
        missing_kwds={
            "color": "lightgrey",
            "edgecolor": "black",
            "label": "Sin datos"
        }
    )

    for _, row in geo_df.iterrows():
        if row["label_point"] is not None:
            ax.text(
                row["label_point"].x,
                row["label_point"].y,
                str(row[label_col]).title(),
                fontsize=7,
                ha="center",
                va="center",
                color="black",
                bbox=dict(
                    facecolor="white",
                    alpha=0.55,
                    edgecolor="none",
                    boxstyle="round,pad=0.15"
                )
            )

    ax.set_title(titulo, fontsize=16, fontweight="bold")
    ax.axis("off")

    ruta_salida = os.path.join(FIG_DIR, nombre_archivo)
    plt.savefig(ruta_salida, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Figura guardada en: {ruta_salida}")


# ============================================================
#  MAPAS ESTÁTICOS PRINCIPALES
# ============================================================

mapa_estatico_localidad(
    geo_df=geo_indicadores,
    columna="peso_promedio",
    titulo="Peso promedio al nacer por localidad de residencia materna",
    nombre_archivo="mapa_peso_promedio_localidad.png",
    cmap="viridis"
)

mapa_estatico_localidad(
    geo_df=geo_indicadores,
    columna="prevalencia_bpn",
    titulo="Prevalencia de bajo peso al nacer por localidad de residencia materna",
    nombre_archivo="mapa_prevalencia_bpn_localidad.png",
    cmap="Reds"
)

mapa_estatico_localidad(
    geo_df=geo_indicadores,
    columna="total_nacimientos",
    titulo="Número de nacimientos registrados por localidad de residencia materna",
    nombre_archivo="mapa_total_nacimientos_localidad.png",
    cmap="Blues"
)


# ============================================================
#  INDICADORES ESPACIO-TEMPORALES POR AÑO Y LOCALIDAD
# ============================================================
# Se calcula:
#   - peso_promedio_{j,t}
#   - prevalencia_bpn_{j,t}
#   - total_nacimientos_{j,t}

df_espacio_tiempo = (
    df
    .dropna(subset=["LOCALIDAD_MADRE_LIMPIA", COL_ANIO])
    .groupby([COL_ANIO, "LOCALIDAD_MADRE_LIMPIA"])
    .agg(
        peso_promedio=(COL_PESO, "mean"),
        prevalencia_bpn=("BPN_NUM", "mean"),
        total_nacimientos=(COL_PESO, "count")
    )
    .reset_index()
)

#display(df_espacio_tiempo.head())


# ============================================================
# MAPAS COMPARATIVOS POR AÑO EN GRILLA
# ============================================================

def mapas_por_anio_grid(
    geo_base,
    df_temporal,
    columna,
    titulo_general,
    nombre_archivo,
    cmap="viridis",
    ncols=3
):
    """
    Genera mapas por año en una sola figura comparativa.
    Cada panel corresponde a un año.
    """

    anios = sorted(df_temporal[COL_ANIO].dropna().unique())
    n_anios = len(anios)
    nrows = int(np.ceil(n_anios / ncols))

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(5.5 * ncols, 5.5 * nrows)
    )

    axes = np.array(axes).reshape(-1)

    vmin = df_temporal[columna].min()
    vmax = df_temporal[columna].max()

    for i, anio in enumerate(anios):
        ax = axes[i]

        temp = df_temporal[df_temporal[COL_ANIO] == anio]

        geo_temp = geo_base.merge(
            temp,
            left_on="LOCALIDAD_LIMPIA",
            right_on="LOCALIDAD_MADRE_LIMPIA",
            how="left"
        )

        geo_temp["label_point"] = geo_temp.geometry.representative_point()

        geo_temp.plot(
            column=columna,
            cmap=cmap,
            linewidth=0.6,
            edgecolor="black",
            ax=ax,
            vmin=vmin,
            vmax=vmax,
            missing_kwds={
                "color": "lightgrey",
                "edgecolor": "black"
            }
        )

        for _, row in geo_temp.iterrows():
            ax.text(
                row["label_point"].x,
                row["label_point"].y,
                str(row["LOCALIDAD_LIMPIA"]).title(),
                fontsize=5.5,
                ha="center",
                va="center",
                color="black",
                bbox=dict(
                    facecolor="white",
                    alpha=0.45,
                    edgecolor="none",
                    boxstyle="round,pad=0.1"
                )
            )

        ax.set_title(f"Año {int(anio)}", fontsize=12, fontweight="bold")
        ax.axis("off")

    # Apagar paneles vacíos
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    # Barra de color común
    sm = plt.cm.ScalarMappable(
        cmap=cmap,
        norm=plt.Normalize(vmin=vmin, vmax=vmax)
    )
    sm._A = []

    cbar = fig.colorbar(
        sm,
        ax=axes,
        fraction=0.025,
        pad=0.02
    )
    cbar.set_label(columna.replace("_", " ").title(), fontsize=11)

    fig.suptitle(titulo_general, fontsize=18, fontweight="bold", y=0.98)

    ruta_salida = os.path.join(FIG_DIR, nombre_archivo)
    plt.savefig(ruta_salida, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Figura guardada en: {ruta_salida}")


mapas_por_anio_grid(
    geo_base=geo_bogota,
    df_temporal=df_espacio_tiempo,
    columna="peso_promedio",
    titulo_general="Peso promedio al nacer por localidad y año",
    nombre_archivo="mapas_peso_promedio_por_anio.png",
    cmap="viridis",
    ncols=3
)

mapas_por_anio_grid(
    geo_base=geo_bogota,
    df_temporal=df_espacio_tiempo,
    columna="prevalencia_bpn",
    titulo_general="Prevalencia de bajo peso al nacer por localidad y año",
    nombre_archivo="mapas_prevalencia_bpn_por_anio.png",
    cmap="Reds",
    ncols=3
)


# ============================================================
#  MAPA INTERACTIVO FOLIUM 
# ============================================================

#geo_indicadores_wgs84 = geo_indicadores.to_crs(epsg=4326).copy()

# Eliminar columnas geométricas auxiliares no serializables
#geo_indicadores_wgs84 = geo_indicadores_wgs84.drop(
#    columns=["label_point"],
#    errors="ignore"
#)

#mapa_folium = folium.Map(
#    location=[4.65, -74.10],
#    zoom_start=10,
#    tiles="cartodbpositron"
#)

#folium.Choropleth(
#    geo_data=geo_indicadores_wgs84.__geo_interface__,
#    data=geo_indicadores_wgs84,
#    columns=["LOCALIDAD_LIMPIA", "peso_promedio"],
#    key_on="feature.properties.LOCALIDAD_LIMPIA",
#    fill_color="YlGnBu",
#    fill_opacity=0.75,
#    line_opacity=0.6,
#    legend_name="Peso promedio al nacer (gramos)",
#    nan_fill_color="lightgray"
#).add_to(mapa_folium)

#folium.GeoJson(
#    geo_indicadores_wgs84.__geo_interface__,
#    tooltip=folium.GeoJsonTooltip(
#        fields=[
#            "LOCALIDAD_LIMPIA",
#            "peso_promedio",
#            "prevalencia_bpn",
#            "total_nacimientos"
#        ],
#        aliases=[
#            "Localidad:",
#            "Peso promedio:",
#            "Prevalencia BPN:",
#            "Total nacimientos:"
#        ],
#        localize=True,
#        sticky=True
#    )
#).add_to(mapa_folium)

#ruta_mapa_folium = os.path.join(
#    FIG_DIR,
#    "mapa_interactivo_peso_promedio_folium.html"
#)

#mapa_folium.save(ruta_mapa_folium)

#print(f"Mapa interactivo guardado en: {ruta_mapa_folium}")

#mapa_folium


# ============================================================
#  MAPA INTERACTIVO PLOTLY 
# ============================================================

geo_plotly = geo_indicadores.copy()

# Eliminar geometrías auxiliares no serializables
#geo_plotly = geo_plotly.drop(
#    columns=["label_point"],
#    errors="ignore"
#)

#geojson_plotly = geo_plotly.__geo_interface__

#fig_plotly = px.choropleth(
#    geo_plotly,
#    geojson=geojson_plotly,
#    locations=geo_plotly.index,
#    color="peso_promedio",
#    hover_name="LOCALIDAD_LIMPIA",
#    hover_data={
#        "peso_promedio": ":.1f",
#        "mediana_peso": ":.1f",
#        "prevalencia_bpn": ":.3f",
#        "total_nacimientos": True
#    },
#    projection="mercator",
#    color_continuous_scale="Viridis",
#    title="Peso promedio al nacer por localidad de residencia materna"
#)

#fig_plotly.update_geos(
#    fitbounds="locations",
#    visible=False
#)

#fig_plotly.update_layout(
#    height=650,
#    margin={"r": 0, "t": 60, "l": 0, "b": 0}
#)

#ruta_plotly = os.path.join(
#    FIG_DIR,
#    "mapa_interactivo_peso_promedio_plotly.html"
#)

#fig_plotly.write_html(ruta_plotly)

#fig_plotly.show()

#print(f"Mapa interactivo Plotly guardado en: {ruta_plotly}")



Se observa que existe una variabilidad espacial observable tanto en el peso promedio al nacer como en la prevalencia de BPN entre localidades de Bogotá. Aunque las diferencias en peso promedio no son extremadamente grandes (aprox. 130–140 gramos entre extremos), sí muestran un gradiente territorial consistente. Por ejemplo, localidades como Sumapaz, Usme, San Cristóbal y Ciudad Bolívar presentan los menores pesos promedio y simultáneamente algunas de las prevalencias más altas de bajo peso al nacer. En contraste, localidades como Teusaquillo, Fontibón, Engativá y Usaquén exhiben mayores pesos promedio y menores prevalencias de BPN. Este patrón es coherente con posibles desigualdades socioeconómicas, diferencias en acceso a servicios de salud, condiciones nutricionales maternas y determinantes sociales asociados al territorio.

También es importante notar que las localidades con mayor volumen de nacimientos (Suba, Kennedy, Ciudad Bolívar y Bosa) concentran una parte importante de la carga poblacional del fenómeno, por lo que pequeñas diferencias porcentuales pueden traducirse en un número absoluto considerable de nacimientos con bajo peso. Por otro lado, localidades con muy pocos registros, como Sumapaz o Candelaria, deben interpretarse con cautela debido a la mayor inestabilidad estadística de sus estimaciones; por ejemplo, la prevalencia de BPN en Sumapaz (~21%) podría estar influenciada por el tamaño muestral reducido ((n=190)).

En términos analíticos, los resultados sugieren que la variable territorial podría aportar información relevante al modelamiento predictivo del peso al nacer, ya sea incorporando directamente la localidad como predictor categórico o mediante variables derivadas que representen contexto territorial. Sin embargo, las diferencias observadas deben interpretarse como asociaciones espaciales descriptivas y no como relaciones causales directas, dado que la localidad probablemente actúa como proxy de múltiples determinantes estructurales y contextuales no observados explícitamente en la base.


## **Análisis para variables cualitativas**
A continuación se consideran las distribuciones de clases para cada una de las variables cualitativas presentes en el estudio.

In [ ]:
# ============================================================
# VARIABLES CUALITATIVAS / CATEGÓRICAS
# ============================================================

variables_cualitativas = [
    'SIT_PARTO',
    'SEXO',
    'PESO',
    'BPN',
    'ANO',
    'ATENCION_PARTO_POR',
    'GESTACION',
    'TIPO_PARTO',
    'MULTIPLICIDAD_PARTO',
    'GRUPO_SANGUINEO',
    'FACTOR_RH',
    'PERTENENCIA_ETNICA',
    'TIPO_DOC_MADRE',
    'GRUPO_QUINQUENAL_MADRE_CALCULADORA',
    'ESTADO_CONYUGAL_MADRE',
    'NIVEL_EDUCATIVO_MADRE',
    'LOCALIDAD_MADRE',
    'REGIMEN_SEGURIDAD',
    'GRUPO_QUINQUENAL_PADRE_CALCULADORA',
    'NIVEL_EDUCATIVO_PADRE'
]

# Filtrar solo las que existen en df
variables_cualitativas = [
    var for var in variables_cualitativas
    if var in df.columns
]

# Lista para guardar tablas resumen
resumen_categoricas = []

# ============================================================
# ANÁLISIS UNIVARIADO CATEGÓRICO
# ============================================================

for var in variables_cualitativas:

    print(f"\nAnálisis para: {var}\n")

    conteo = df[var].value_counts(dropna=False)
    total = len(df)
    nulos = df[var].isnull().sum()

    resumen = (
        pd.DataFrame({
            'Categoría': conteo.index.astype(str),
            'Conteo': conteo.values,
            'Porcentaje (%)': (conteo.values / total * 100).round(2)
        })
        .sort_values('Conteo', ascending=False)
        .reset_index(drop=True)
    )

    resumen['Variable'] = var

    resumen_categoricas.append(resumen)

    display(resumen[['Categoría', 'Conteo', 'Porcentaje (%)']])
    print(f"Valores nulos: {nulos}\n")

    fig = px.bar(
        resumen,
        x='Categoría',
        y='Conteo',
        color='Categoría',
        text='Conteo',
        title=f'Distribución de {var}',
        labels={
            'Categoría': var,
            'Conteo': 'Número de registros'
        }
    )

    fig.update_traces(
        textposition='outside'
    )

    fig.update_layout(
        autosize=True,
        height=500,
        template='plotly_white',
        showlegend=False,
        margin=dict(l=40, r=40, t=80, b=80),
        xaxis_tickangle=-45,
        xaxis_title='',
        yaxis_title='Número de registros'
    )

    fig.show(config={'responsive': True})

# ============================================================
# TABLA CONSOLIDADA FINAL
# ============================================================

df_resumen_categoricas = pd.concat(
    resumen_categoricas,
    ignore_index=True
)

#print("\nResumen consolidado de variables categóricas:\n")
#display(df_resumen_categoricas)

A nivel descriptivo, la cohorte analizada presenta patrones demográficos, obstétricos y neonatales coherentes con una población predominantemente urbana y con atención institucionalizada del parto. En primer lugar, prácticamente la totalidad de los nacimientos ocurrieron en instituciones de salud y fueron atendidos por personal médico ((\approx 99.7%)), lo que sugiere una alta cobertura de atención profesional del parto dentro de la población estudiada. Asimismo, la distribución por sexo mostró una proporción equilibrada entre nacimientos masculinos (51.14%) y femeninos (48.85%), sin diferencias relevantes desde el punto de vista poblacional.

Respecto a las características neonatales, la mayoría de los recién nacidos se clasificaron dentro de rangos de peso considerados normales, predominando la categoría `5.NORMAL (3000–4199 g)` con aproximadamente el 48.5% de los registros, seguida por `4.DEFICIT (2500–2999 g)` con 36.1%. Aunque la proporción de pesos extremadamente altos fue mínima ((<1%)), se observó que cerca del 13.7% de los nacimientos correspondieron a categorías de bajo peso (`1500–2499 g`). De forma consistente, la prevalencia global de bajo peso al nacer (BPN) fue del 15.25%, mientras que el 84.75% de los nacimientos no presentaron esta condición. Adicionalmente, se identificó una disminución progresiva en el número de nacimientos a lo largo del periodo analizado, observándose una reducción relativa desde aproximadamente el 20.6% de los registros en 2020 hasta cerca del 14.5% en 2025.

Desde la perspectiva obstétrica, predominó ampliamente la gestación a término (`37–42 semanas`) con un 88.2%, mientras que los nacimientos pretérmino representaron alrededor del 11.8%, proporción clínicamente relevante dada su estrecha asociación con el bajo peso al nacer. En cuanto al tipo de parto, se observó una distribución prácticamente equilibrada entre parto espontáneo (48.9%) y cesárea (48.5%), con una baja frecuencia de partos instrumentados (2.6%). Asimismo, la mayoría de los embarazos correspondieron a gestaciones simples (98%), siendo poco frecuentes los embarazos múltiples.

A nivel sociodemográfico, se evidenció una marcada concentración en determinadas categorías. Territorialmente, localidades como Suba, Kennedy, Ciudad Bolívar y Bosa concentraron el mayor volumen de nacimientos. En relación con el aseguramiento, predominó el régimen contributivo (68.4%), seguido del subsidiado (21.1%). La mayoría de las madres presentaron educación media académica o profesional, mientras que en estado conyugal predominó la convivencia en pareja, particularmente bajo la categoría “no casada con convivencia ≥2 años”. En cuanto a pertenencia étnica, más del 99% de los registros se clasificaron en “ninguno de los anteriores”, evidenciando una distribución altamente desbalanceada.

En conjunto, los resultados muestran una fuerte heterogeneidad proporcional tanto intra como entre variables categóricas. Varias categorías presentan frecuencias extremadamente bajas (por ejemplo, categorías étnicas minoritarias, tipos documentales específicos, embarazos cuádruples o más, entre otras), lo cual puede generar inestabilidad estadística y problemas de representación durante el modelamiento. Por esta razón, será metodológicamente recomendable realizar procesos de agrupación o recategorización de niveles poco frecuentes, especialmente en variables cualitativas de alta cardinalidad o con distribuciones altamente desbalanceadas, con el fin de mejorar la robustez, estabilidad e interpretabilidad de los modelos predictivos posteriores.


## **Análisis univariados desagregados por año para variables cuantitativas**

In [ ]:
# Filtrar variables existentes
variables_cualitativas = [
    'SIT_PARTO',
    'SEXO',
    'PESO',
    'BPN',
    'ATENCION_PARTO_POR',
    'GESTACION',
    'TIPO_PARTO',
    'MULTIPLICIDAD_PARTO',
    'PERTENENCIA_ETNICA',
    'TIPO_DOC_MADRE',
    'GRUPO_QUINQUENAL_MADRE_CALCULADORA',
    'ESTADO_CONYUGAL_MADRE',
    'NIVEL_EDUCATIVO_MADRE',
    'LOCALIDAD_MADRE',
    'REGIMEN_SEGURIDAD',
    'NIVEL_EDUCATIVO_PADRE'
]
variables_cualitativas = [
    var for var in variables_cualitativas
    if var in df.columns
]

# ============================================================
# LISTA PARA CONSOLIDAR RESULTADOS
# ============================================================

resumen_categoricas_anual = []

# ============================================================
# ANÁLISIS DESAGREGADO POR AÑO
# ============================================================

for var in variables_cualitativas:

    print("\n" + "="*80)
    print(f"ANÁLISIS PARA: {var}")
    print("="*80)

    # --------------------------------------------------------
    # TABLA RESUMEN
    # --------------------------------------------------------

    df_temp = (
        df[[col_anio, var]]
        .copy()
    )

    resumen = (
        df_temp
        .groupby([col_anio, var], dropna=False)
        .size()
        .reset_index(name='Conteo')
    )

    total_anual = (
        df_temp
        .groupby(col_anio)
        .size()
        .reset_index(name='Total_anual')
    )

    resumen = resumen.merge(
        total_anual,
        on=col_anio,
        how='left'
    )

    resumen['Porcentaje (%)'] = (
        resumen['Conteo'] /
        resumen['Total_anual'] * 100
    ).round(2)

    resumen['Variable'] = var

    resumen_categoricas_anual.append(resumen)

    display(
        resumen.sort_values(
            [col_anio, 'Conteo'],
            ascending=[True, False]
        )
    )

    # --------------------------------------------------------
    # GRÁFICO DESAGREGADO POR AÑO
    # --------------------------------------------------------

    fig = px.bar(
        resumen,
        x='ANO',
        y='Conteo',
        color=var,
        barmode='group',
        text='Conteo',
        title=f'Distribución de {var} por año',
        labels={
            'ANO': 'Año',
            'Conteo': 'Número de registros'
        }
    )

    fig.update_traces(
        textposition='outside'
    )

    fig.update_layout(
        autosize=True,
        height=550,
        template='plotly_white',
        margin=dict(l=40, r=40, t=80, b=80),
        xaxis_title='Año',
        yaxis_title='Número de registros'
    )

    fig.show(config={'responsive': True})

# ============================================================
# TABLA CONSOLIDADA FINAL
# ============================================================

df_resumen_categoricas_anual = pd.concat(
    resumen_categoricas_anual,
    ignore_index=True
)

In [ ]:
# ============================================================
# AGRUPACIÓN Y RECODIFICACIÓN DE VARIABLES CATEGÓRICAS
# PROYECTO BPN - PESO AL NACER
# ============================================================

# ============================================================
#  AREA DE NACIMIENTO
# ============================================================

map_localidad_nacimiento = {

    # CENTRO
    "Teusaquillo": "Centro",
    "Barrios Unidos": "Centro",
    "Los Mártires": "Centro",
    "Santa Fe": "Centro",
    "La Candelaria": "Centro",

    # NORTE
    "Usaquén": "Norte",
    "Chapinero": "Norte",

    # SURORIENTE
    "San Cristóbal": "Suroriente",
    "Rafael Uribe Uribe": "Suroriente",
    "Tunjuelito": "Suroriente",
    "Usme": "Suroriente",
    "Antonio Nariño": "Suroriente",

    # NOROCCIDENTE
    "Suba": "Noroccidente",
    "Engativá": "Noroccidente",

    # OCCIDENTE
    "Kennedy": "Occidente",
    "Bosa": "Occidente",
    "Fontibón": "Occidente",
    "Puente Aranda": "Occidente",
    "UNETE ARANDA": "Occidente",

    # SUR
    "Ciudad Bolívar": "Sur",
    "Sumapaz": "Sur",

    # DESCONOCIDA
    "Bogotá": "Desconocida",
    "Localidad Desconocida": "Desconocida"
}

df["AREA_NACIMIENTO"] = (
    df["LOCALIDAD_NACIMIENTO"]
    .map(map_localidad_nacimiento)
    .fillna("Desconocida")
)

# ============================================================
#  AREA DE RESIDENCIA MATERNA
# ============================================================

map_localidad_madre = {

    "Teusaquillo": "Centro",
    "Barrios Unidos": "Centro",
    "Los Mártires": "Centro",
    "Santa Fe": "Centro",
    "La Candelaria": "Centro",

    "Usaquén": "Norte",
    "Chapinero": "Norte",

    "San Cristóbal": "Suroriente",
    "Rafael Uribe Uribe": "Suroriente",
    "Tunjuelito": "Suroriente",
    "Usme": "Suroriente",
    "Antonio Nariño": "Suroriente",

    "Suba": "Noroccidente",
    "Engativá": "Noroccidente",

    "Kennedy": "Occidente",
    "Bosa": "Occidente",
    "Fontibón": "Occidente",
    "Puente Aranda": "Occidente",

    "Ciudad Bolívar": "Sur",
    "Sumapaz": "Sur",

    "Localidad Desconocida": "Desconocida"
}

df["AREA_RESIDENCIA_MADRE"] = (
    df["LOCALIDAD_MADRE"]
    .map(map_localidad_madre)
    .fillna("Desconocida")
)

# ============================================================
#  ATENCIÓN DEL PARTO
# ============================================================

df["ATENCION_PARTO_GRUPO"] = np.where(
    df["ATENCION_PARTO_POR"].isin([
        "1.MEDICO",
        "2.ENFERMERO(A)",
        "3.AUXILIAR DE ENFERMERIA",
        "4.PROMOTORA DE SALUD",
        "4.PROMOTOR(A) DE SALUD"
    ]),
    "Profesional de salud",
    "No profesional"
)

# ============================================================
#  SEXO
# ============================================================

df["SEXO_AGRUPADO"] = (
    df["SEXO"]
    .replace({
        "INDETERMINADO": "FEMENINO"
    })
)

# ============================================================
#  MULTIPLICIDAD DEL PARTO
# ============================================================

df["MULTIPLICIDAD_AGRUPADA"] = np.where(
    df["MULTIPLICIDAD_PARTO"] == "SIMPLE",
    "Simple",
    "Multiple"
)

# ============================================================
#  PERTENENCIA ÉTNICA
# ============================================================

df["PERTENENCIA_ETNICA_AGRUPADA"] = np.where(
    df["PERTENENCIA_ETNICA"] ==
    "6.NINGUNO DE LOS ANTERIORES",
    "No pertenece a grupo etnico reconocido",
    "Grupo etnico reconocido"
)

# ============================================================
#  TIPO DOCUMENTO MADRE
# ============================================================

def agrupar_documento(x):

    if pd.isna(x):
        return "Sin identificacion / informacion"

    if x == "CC":
        return "Colombiana identificada"

    if x in ["TI", "RC"]:
        return "Menor identificada"

    if x in [
        "CE",
        "DOCUMENTO EXTRANJERO",
        "PERMISO POR PROTECCION TEMPORAL",
        "PEP",
        "PA",
        "PASAPORTE DE LA ONU",
        "CARNET DIPLOMATICO",
        "SALVOCONDUCTO DE PERMANENCIA"
    ]:
        return "Extranjera"

    return "Sin identificacion / informacion"


df["TIPO_DOC_MADRE_AGRUPADO"] = (
    df["TIPO_DOC_MADRE"]
    .apply(agrupar_documento)
)

# ============================================================
#  ESTADO CONYUGAL MADRE
# ============================================================

def agrupar_estado_conyugal(x):

    if x in ["CASADA", "ESTABA CASADO(A)"]:
        return "Casada"

    if x in [
        "NO CASADA 2 AÑOS O MAS VIVIENDO CON PAREJA",
        "NO ESTABA CASADO(A) Y LLEVABA DOS AÑOS O MAS VIVIENDO CON SU PAREJA",
        "NO CASADA MENOS DE 2 AÑOS VIVIENDO CON PAREJA"
    ]:
        return "Union libre"

    return "Sin pareja"


df["ESTADO_CONYUGAL_AGRUPADO"] = (
    df["ESTADO_CONYUGAL_MADRE"]
    .apply(agrupar_estado_conyugal)
)

# ============================================================
#  NIVEL EDUCATIVO MADRE
# ============================================================

def agrupar_educacion(x):

    if x in [
        "01.PREESCOLAR",
        "13.NINGUNO",
        "02.BASICA PRIMARIA",
        "03.BASICA SECUNDARIA",
        "99.SIN INFORMACION"
    ]:
        return "Basica"

    if x in [
        "04.MEDIA ACADEMICA",
        "05.MEDIA TECNICA",
        "06.NORMALISTA"
    ]:
        return "Media"

    if x in [
        "07.TECNICA PROFESIONAL",
        "08.TECNOLOGICA",
        "09.PROFESIONAL"
    ]:
        return "Superior"

    if x in [
        "10.ESPECIALIZACION",
        "11.MAESTRIA",
        "12.DOCTORADO"
    ]:
        return "Posgrado"

    return np.nan


df["NIVEL_EDUCATIVO_MADRE_AGRUPADO"] = (
    df["NIVEL_EDUCATIVO_MADRE"]
    .apply(agrupar_educacion)
)

# ============================================================
#  NIVEL EDUCATIVO PADRE
# ============================================================

df["NIVEL_EDUCATIVO_PADRE_AGRUPADO"] = (
    df["NIVEL_EDUCATIVO_PADRE"]
    .apply(agrupar_educacion)
)

# ============================================================
#  REGIMEN DE SEGURIDAD
# ============================================================

def agrupar_regimen(x):

    if x == "CONTRIBUTIVO":
        return "Contributivo"

    if x == "SUBSIDIADO":
        return "Subsidiado"

    if x in ["NO ASEGURADO", "SIN INFORMACION"]:
        return "No asegurado"

    return "Especial_Excepcion"


df["REGIMEN_SEGURIDAD_AGRUPADO"] = (
    df["REGIMEN_SEGURIDAD"]
    .apply(agrupar_regimen)
)

# ============================================================
#  APGAR 1
# ============================================================

def limpiar_apgar(x):

    if pd.isna(x):
        return np.nan

    if str(x).upper() == "SIN INFORMACION":
        return np.nan

    try:
        return int(float(x))
    except:
        return np.nan


df["APGAR1_NUM"] = df["APGAR1"].apply(limpiar_apgar)

# imputar moda
moda_apgar1 = df["APGAR1_NUM"].mode()[0]

df["APGAR1_NUM"] = (
    df["APGAR1_NUM"]
    .fillna(moda_apgar1)
)

def categorizar_apgar(x):

    if x <= 3:
        return "Bajo"

    elif x <= 6:
        return "Medio"

    else:
        return "Alto"


df["APGAR1_AGRUPADO"] = (
    df["APGAR1_NUM"]
    .apply(categorizar_apgar)
)

# ============================================================
# VERIFICACIÓN
# ============================================================

variables_nuevas = [
    "AREA_NACIMIENTO",
    "AREA_RESIDENCIA_MADRE",
    "ATENCION_PARTO_GRUPO",
    "SEXO_AGRUPADO",
    "MULTIPLICIDAD_AGRUPADA",
    "PERTENENCIA_ETNICA_AGRUPADA",
    "TIPO_DOC_MADRE_AGRUPADO",
    "ESTADO_CONYUGAL_AGRUPADO",
    "NIVEL_EDUCATIVO_MADRE_AGRUPADO",
    "NIVEL_EDUCATIVO_PADRE_AGRUPADO",
    "REGIMEN_SEGURIDAD_AGRUPADO",
    "APGAR1_AGRUPADO"
]

for v in variables_nuevas:
    print(f"\n{'='*70}")
    print(v)
    print(df[v].value_counts(dropna=False))

In [ ]:
df.to_csv("../data/processed/nacimientos_bogota_preprocesado.csv", index=False, sep=";", encoding="ISO-8859-1") 

## **Matriz de correlación**

In [ ]:
df_corr = df[variables_cuantitativas]

# Calcular matriz de correlación (Pearson)
matriz_corr = df_corr.corr().round(3)

# Mostrar la matriz como tabla
print("\n Matriz de correlación entre variables numéricas:\n")
display(matriz_corr)